# Local Training - Federated MNIST

This notebook trains local client models for a federated learning simulation.

Important correction: all clients start from the same initial global model. This is necessary because FedAvg assumes that local models are updates from a shared global initialization.


In [ ]:
import os
import sys
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath("..")
SRC_PATH = os.path.join(PROJECT_ROOT, "src")

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

from TheModel import build_model
from utils import load_client_data, plot_learning_curves, evaluate_model

print("Project root:", PROJECT_ROOT)
print("Source path:", SRC_PATH)
print("TensorFlow version:", tf.__version__)


In [ ]:
# Experiment configuration

N_CLIENTS = 3
EPOCHS = 3
BATCH_SIZE = 64
VALIDATION_SPLIT = 0.1
SEED = 42

LOCAL_MODELS_DIR = os.path.join(PROJECT_ROOT, "local_models")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
GLOBAL_MODELS_DIR = os.path.join(PROJECT_ROOT, "global_models")

os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(GLOBAL_MODELS_DIR, exist_ok=True)

tf.keras.utils.set_random_seed(SEED)


In [ ]:
# Create and save one shared initial global model

initial_global_path = os.path.join(GLOBAL_MODELS_DIR, "global_initial.keras")

initial_global_model = build_model()
initial_global_model.save(initial_global_path)

print("Saved shared initial global model to:", initial_global_path)


In [ ]:
# Local training loop

local_results = []
client_sample_counts = {}

for client_id in range(1, N_CLIENTS + 1):
    print("=" * 80)
    print(f"Training local model for client {client_id}")
    print("=" * 80)

    x_train, y_train, x_test, y_test = load_client_data(
        client_id=client_id,
        data_dir=os.path.join(PROJECT_ROOT, "local_data")
    )

    client_sample_counts[f"client_{client_id}"] = int(len(x_train))

    print("x_train shape:", x_train.shape)
    print("y_train shape:", y_train.shape)

    # Every client starts from the exact same global initialization.
    model = tf.keras.models.load_model(initial_global_path)

    history = model.fit(
        x_train,
        y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_split=VALIDATION_SPLIT,
        verbose=1
    )

    plot_learning_curves(
        history,
        title=f"Client {client_id} Local Training"
    )

    test_loss, test_accuracy = evaluate_model(model, x_test, y_test)

    model_path = os.path.join(
        LOCAL_MODELS_DIR,
        f"client_{client_id}_local.keras"
    )

    model.save(model_path)

    local_results.append({
        "client_id": client_id,
        "samples": int(len(x_train)),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_accuracy),
        "final_train_accuracy": float(history.history["accuracy"][-1]),
        "final_val_accuracy": float(history.history["val_accuracy"][-1]),
        "final_train_loss": float(history.history["loss"][-1]),
        "final_val_loss": float(history.history["val_loss"][-1]),
        "model_path": model_path
    })

    print(f"Saved local model to: {model_path}")


In [ ]:
# Save local training results and metadata

results_path = os.path.join(RESULTS_DIR, "local_training_results.json")
metadata_path = os.path.join(RESULTS_DIR, "client_sample_counts.json")

with open(results_path, "w") as f:
    json.dump(local_results, f, indent=4)

with open(metadata_path, "w") as f:
    json.dump(client_sample_counts, f, indent=4)

print("Saved results to:", results_path)
print("Saved client sample counts to:", metadata_path)

local_results


## Interpretation

Each client trained a local model using only its own private partition of MNIST. Since all clients started from the same global initialization, their resulting weights can be meaningfully aggregated using FedAvg-style methods.
